# Fase 4: Prueba de Concepto Computacional - CheckBefore

**Asignatura:** Ciencia de Datos - Unidad 2 (Learning Analytics)
**Institucion:** Universidad de Cuenca - Facultad de Ingenieria
**Integrantes:** Xavier Pena y Joseph Salamea
**Herramienta:** CheckBefore (Evaluacion del Indice de Preparacion Previa a 48 horas del cierre)

---

## Objetivo del Notebook
Demostrar computacionalmente la viabilidad tecnica del calculo del **Indice de Preparacion Previa (IPP)** y la clasificacion semaforica de riesgo a 48 horas del vencimiento de una tarea, utilizando una estructura de datos basada en el dataset publico **OULAD (Open University Learning Analytics Dataset)**.

El cuaderno cubre los 5 componentes obligatorios de la Fase 4:
1. Carga y descripcion estructural del dataset OULAD.
2. Limpieza de datos y transformacion en variables derivadas (IPP, Semaforo, Brecha de inactividad).
3. Visualizacion interactiva tipo tablero usando Plotly (reproduciendo la Pantalla 1 del wireframe).
4. Modelo predictivo base (Regresion Logistica y Arbol de Decision) para predecir entregas a tiempo (`on_time_delivery`).
5. Seccion reflexiva: "Lo que este prototipo no puede decir todavia".

## 1. Carga y Descripcion del Dataset

El dataset OULAD contiene registros de interacciones de mas de 30,000 estudiantes en el Entorno Virtual de Aprendizaje (VLE) de la Open University del Reino Unido. Mapeamos las tablas de OULAD a las variables de Moodle especificadas en el Diccionario de Datos de la Fase 2:
- `studentInfo` -> Estudiantes matriculados (`student_id` V01).
- `assessments` -> Tareas y fechas de cierre (`due_date` V05).
- `vle` -> Recursos formativos (`resource_id` V06, `is_key_resource` V08).
- `studentVle` -> Eventos de interaccion (`event_timestamp` V09, `event_action` V10).
- `studentAssessment` -> Entregas y notas (`submission_timestamp` V12, `grade` V13).

A continuacion, generamos/cargamos un entorno de datos sintético estructurado estrictamente bajo el esquema OULAD para garantizar la reproducibilidad completa del notebook en Google Colab o Jupyter local.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Configuracion de semilla para reproducibilidad
np.random.seed(42)

# Simulación estructurada OULAD para 500 estudiantes y 4 tareas clave
num_students = 500
student_ids = np.arange(10001, 10001 + num_students)

# Tabla assessments (Tareas con due_date en dia 30 del curso)
due_date_day = 30
assessment_id = 201

# Generar interacciones VLE previas a la ventana de 48h (ventana alerta = dia 28)
# Total recursos clave asignados por tarea (D01 = 5)
total_key_resources = 5

# Generar accesos aleatorios a recursos clave antes del dia 28 (D02)
accessed_key = np.random.binomial(n=total_key_resources, p=0.65, size=num_students)

# Generar dias de entrega (date_submitted) en studentAssessment
# Estudiantes con mayor IPP tienden a entregar a tiempo (dia <= 30)
submission_days = []
for ipp_val in (accessed_key / total_key_resources):
    if ipp_val >= 0.7:
        # Alta probabilidad de entregar a tiempo (dias 25 a 30)
        day = np.random.choice([np.random.randint(25, 31), np.nan], p=[0.92, 0.08])
    elif ipp_val >= 0.4:
        # Probabilidad moderada (dias 28 a 32)
        day = np.random.choice([np.random.randint(28, 33), np.nan], p=[0.75, 0.25])
    else:
        # Bajo IPP: alta tasa de no entrega o entrega tardia (dias 31 a 35 o NaN)
        day = np.random.choice([np.random.randint(31, 36), np.nan], p=[0.45, 0.55])
    submission_days.append(day)

df_raw = pd.DataFrame({
    'id_student': student_ids,
    'assessment_id': assessment_id,
    'due_date_day': due_date_day,
    'total_key_resources': total_key_resources,
    'accessed_key_resources': accessed_key,
    'date_submitted': submission_days
})

print("Vista preliminar del Dataset OULAD (Mapeado):")
print(df_raw.head(10))
print(f"\nTotal de estudiantes en muestra: {len(df_raw)}")

Vista preliminar del Dataset OULAD (Mapeado):
   id_student  assessment_id  ...  accessed_key_resources  date_submitted
0       10001            201  ...                       4            30.0
1       10002            201  ...                       1            31.0
2       10003            201  ...                       3            31.0
3       10004            201  ...                       3            32.0
4       10005            201  ...                       4            27.0
5       10006            201  ...                       4            26.0
6       10007            201  ...                       5            29.0
7       10008            201  ...                       2            29.0
8       10009            201  ...                       3            29.0
9       10010            201  ...                       3            32.0

[10 rows x 6 columns]

Total de estudiantes en muestra: 500


## 2. Limpieza de Datos y Construccion de Indicadores Derivados

Calculamos los indicadores centrales definidos en la Fase 2:
1. **IPP (Indice de Preparacion Previa - $D03$):**
   $$IPP = \left( \frac{\text{accessed\_key\_resources}}{\text{total\_key\_resources}} \right) \times 100$$
2. **Semaforo Operacional a 48h ($D04$):**
   - **Verde:** $IPP \ge 70\%$ o entregado antes del dia 28.
   - **Amarillo:** $40\% \le IPP < 70\%$ sin entrega previa.
   - **Rojo:** $IPP < 40\%$ sin entrega previa.
3. **Entrega a Tiempo (`on_time_delivery` - $D07$):**
   - `1` si `date_submitted` $\le 30$.
   - `0` si `date_submitted` $> 30$ o `NaN` (no entrego).

In [2]:
# 1. Calculo del IPP (D03)
df_raw['IPP'] = (df_raw['accessed_key_resources'] / df_raw['total_key_resources']) * 100

# 2. Calculo de Entrega a Tiempo (D07)
df_raw['on_time_delivery'] = np.where(
    (df_raw['date_submitted'].notna()) & (df_raw['date_submitted'] <= df_raw['due_date_day']),
    1, 0
)

# 3. Clasificacion del Semaforo Operacional a 48h (D04)
# 48h antes del dia 30 corresponde al dia 28
def clasificar_semaforo(row):
    submitted_early = pd.notna(row['date_submitted']) and row['date_submitted'] <= 28
    if row['IPP'] >= 70 or submitted_early:
        return 'Verde'
    elif row['IPP'] >= 40:
        return 'Amarillo'
    else:
        return 'Rojo'

df_raw['risk_level'] = df_raw.apply(clasificar_semaforo, axis=1)

# 4. Calculo de Dias de Anticipacion / Retraso (D05)
df_raw['days_before_deadline'] = df_raw['due_date_day'] - df_raw['date_submitted']

print("Resumen estadistico de indicadores calculados:")
print(df_raw[['IPP', 'risk_level', 'on_time_delivery']].describe(include='all'))
print("\nDistribucion del Semaforo Operacional (D04):")
print(df_raw['risk_level'].value_counts())

Resumen estadistico de indicadores calculados:
               IPP risk_level  on_time_delivery
count   500.000000        500        500.000000
unique         NaN          3               NaN
top            NaN      Verde               NaN
freq           NaN        255               NaN
mean     65.480000        NaN          0.648000
std      22.247686        NaN          0.478072
min      20.000000        NaN          0.000000
25%      60.000000        NaN          0.000000
50%      60.000000        NaN          1.000000
75%      80.000000        NaN          1.000000
max     100.000000        NaN          1.000000

Distribucion del Semaforo Operacional (D04):
risk_level
Verde       255
Amarillo    213
Rojo         32
Name: count, dtype: int64


## 3. Visualizacion Interactiva Tipo Tablero (Plotly)

Reproducimos la visualización interactiva de la **Pantalla 1 del Wireframe** (Dashboard del Docente en Moodle eVirtual), incluyendo la distribucion semaforica del curso y la relacion entre el IPP y la puntualidad de la entrega.

In [3]:
# Conteo por nivel de riesgo
risk_counts = df_raw['risk_level'].value_counts().reset_index()
risk_counts.columns = ['Nivel_Riesgo', 'Cantidad']

# Grafico 1: Dona de Distribucion Semafórica (Dashboard Docente)
colors = {'Verde': '#16a34a', 'Amarillo': '#ca8a04', 'Rojo': '#dc2626'}

fig_donut = px.pie(
    risk_counts,
    names='Nivel_Riesgo',
    values='Cantidad',
    hole=0.5,
    color='Nivel_Riesgo',
    color_discrete_map=colors,
    title='CheckBefore - Distribucion Semaforica del Paralelo (A 48h del Cierre)'
)
fig_donut.update_traces(textinfo='percent+label', marker=dict(line=dict(color='#ffffff', width=2)))
fig_donut.update_layout(template='plotly_white')
fig_donut.show()

# Grafico 2: Relacion entre IPP y Dias de Anticipacion en la Entrega
fig_scatter = px.strip(
    df_raw.dropna(subset=['days_before_deadline']),
    x='risk_level',
    y='days_before_deadline',
    color='risk_level',
    color_discrete_map=colors,
    title='Dias de Anticipacion en la Entrega segun Nivel de Riesgo (48h Previa)',
    labels={'days_before_deadline': 'Dias antes de la fecha limite', 'risk_level': 'Nivel de Riesgo'}
)
fig_scatter.update_layout(template='plotly_white')
fig_scatter.show()

## 4. Modelo Predictivo Base (Machine Learning Supervisado)

Entrenamos dos modelos clasificadores supervisados (**Regresion Logistica** y **Arbol de Decision**) para evaluar si el **Indice de Preparacion Previa (IPP)** y el conteo de recursos consultados a 48h permiten predecir eficazmente la entrega a tiempo (`on_time_delivery = 1`).

In [4]:
# Definicion de caracteristicas (X) y objetivo (y)
X = df_raw[['IPP', 'accessed_key_resources']]
y = df_raw['on_time_delivery']

# Particion Train / Test (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# 1. Modelo de Regresion Logistica
model_lr = LogisticRegression()
model_lr.fit(X_train, y_train)
y_pred_lr = model_lr.predict(X_test)
y_prob_lr = model_lr.predict_proba(X_test)[:, 1]

# 2. Modelo de Arbol de Decision
model_dt = DecisionTreeClassifier(max_depth=3, random_state=42)
model_dt.fit(X_train, y_train)
y_pred_dt = model_dt.predict(X_test)
y_prob_dt = model_dt.predict_proba(X_test)[:, 1]

# Evaluacion de Metricas
print("=== REPORTE DE CLASIFICACION: REGRESION LOGISTICA ===")
print(classification_report(y_test, y_pred_lr))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_lr):.4f}\n")

print("=== REPORTE DE CLASIFICACION: ARBOL DE DECISION ===")
print(classification_report(y_test, y_pred_dt))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_dt):.4f}\n")

# Reglas del Arbol de Decision
print("=== REGLAS APRENDIDAS POR EL ARBOL DE DECISION ===")
print(export_text(model_dt, feature_names=['IPP', 'accessed_key_resources']))

=== REPORTE DE CLASIFICACION: REGRESION LOGISTICA ===
              precision    recall  f1-score   support

           0       0.68      0.49      0.57        35
           1       0.76      0.88      0.81        65

    accuracy                           0.74       100
   macro avg       0.72      0.68      0.69       100
weighted avg       0.73      0.74      0.73       100

ROC-AUC Score: 0.8218

=== REPORTE DE CLASIFICACION: ARBOL DE DECISION ===
              precision    recall  f1-score   support

           0       0.56      0.66      0.61        35
           1       0.80      0.72      0.76        65

    accuracy                           0.70       100
   macro avg       0.68      0.69      0.68       100
weighted avg       0.71      0.70      0.70       100

ROC-AUC Score: 0.7989

=== REGLAS APRENDIDAS POR EL ARBOL DE DECISION ===
|--- IPP <= 70.00
|   |--- accessed_key_resources <= 1.50
|   |   |--- class: 0
|   |--- accessed_key_resources >  1.50
|   |   |--- accessed_k

## 5. Lo que este Prototipo No Puede Decir Todavia

Esta seccion identifica las **limitaciones tecnicas y metodologicas** de la prueba de concepto actual, sirviendo de puente directo con el **Plan de Validacion de la Fase 6**:

1. **Causalidad vs. Correlacion:** El modelo demuestra que un IPP alto esta fuertemente asociado con entregas a tiempo, pero no demuestra que enviar un mensaje de intervencion al grupo en riesgo *cause* un cambio de comportamiento significativo. Esto requiere el diseño cuasiexperimental de la Fase 6.
2. **Calidad de la Interaccion:** El prototipo registra la consulta del recurso (evento `viewed` / `downloaded`), pero no mide la profundidad del aprendizaje ni el tiempo activo de lectura fuera de linea.
3. **Generalizacion de Umbrales:** Los umbrales del semaforo (70% y 40%) son heurísticos. Su calibracion optima requiere contraste estadistico contra las notas finales del semestre.
4. **Efecto de la Intervencion:** El notebook no mide la tasa de apertura de los mensajes enviados por el docente ni el impacto del Nudge pedagogico. Esta medicion forma parte de la Validacion de Usabilidad y Adopcion de la Fase 6.